# База данных микроклимата

В базе хранятся комната, узел ESP32, датчик DHT22, каналы температуры и влажности, измерения, пороги и предупреждения. UML-схема находится в README.md.

Все файлы лежат в одной папке. Выбери ядро microclimate и запускай ячейки сверху вниз.

## 1. Подключение
Если базы ещё нет, SQLite создаст её в текущей папке. Перед запуском проверь путь, который выводится ниже.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sqlite3

if "conn" in globals():
    conn.close()

db_path = Path("microclimate.db").resolve()
conn = sqlite3.connect(str(db_path))
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")

print("База:", db_path)

## 2. Создание таблиц
CREATE TABLE IF NOT EXISTS позволяет повторно выполнить ячейку. Уже существующие таблицы и данные сохраняются.

In [ ]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS room (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    area_m2 REAL NOT NULL CHECK (area_m2 > 0)
);

CREATE TABLE IF NOT EXISTS node (
    id INTEGER PRIMARY KEY,
    room_id INTEGER NOT NULL REFERENCES room(id),
    device_uid TEXT NOT NULL UNIQUE,
    period_ms INTEGER NOT NULL CHECK (period_ms > 0)
);

CREATE TABLE IF NOT EXISTS sensor (
    id INTEGER PRIMARY KEY,
    node_id INTEGER NOT NULL REFERENCES node(id),
    model TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS channel (
    id INTEGER PRIMARY KEY,
    sensor_id INTEGER NOT NULL REFERENCES sensor(id),
    metric TEXT NOT NULL CHECK (
        metric IN ('temperature', 'humidity')
    ),
    unit TEXT NOT NULL CHECK (
        (metric = 'temperature' AND unit = '°C')
        OR
        (metric = 'humidity' AND unit = '%')
    ),
    UNIQUE (sensor_id, metric)
);

CREATE TABLE IF NOT EXISTS measurement (
    channel_id INTEGER NOT NULL REFERENCES channel(id),
    ts TEXT NOT NULL,
    value REAL NOT NULL,
    PRIMARY KEY (channel_id, ts)
);

CREATE TABLE IF NOT EXISTS threshold (
    id INTEGER PRIMARY KEY,
    channel_id INTEGER NOT NULL REFERENCES channel(id),
    lower_bound REAL NOT NULL,
    upper_bound REAL NOT NULL,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    UNIQUE (id, channel_id),
    CHECK (lower_bound < upper_bound),
    CHECK (valid_to IS NULL OR valid_from < valid_to)
);

CREATE TABLE IF NOT EXISTS alert (
    id INTEGER PRIMARY KEY,
    threshold_id INTEGER NOT NULL,
    channel_id INTEGER NOT NULL,
    ts TEXT NOT NULL,
    direction TEXT NOT NULL CHECK (
        direction IN ('below', 'above')
    ),
    UNIQUE (channel_id, ts),
    FOREIGN KEY (channel_id, ts)
        REFERENCES measurement(channel_id, ts),
    FOREIGN KEY (threshold_id, channel_id)
        REFERENCES threshold(id, channel_id)
);

""")
print("Таблицы созданы.")

## 3. Начальные данные и пороги
Повторный запуск не добавляет дубликаты. Для комнаты используем площадь 12 м², для узла — интервал 2000 мс. Границы 20–26 °C и 40–60 % выбраны для проверки программы, это не нормативы. Существующие пороги не меняем.

In [ ]:
with conn:
    conn.execute(
        "INSERT INTO room (name, area_m2) VALUES (?, ?) ON CONFLICT (name) DO NOTHING",
        ("Учебная комната", 12.0)
    )
    room_id = conn.execute(
        "SELECT id FROM room WHERE name = ?", ("Учебная комната",)
    ).fetchone()[0]

    conn.execute("""
        INSERT INTO node (room_id, device_uid, period_ms) VALUES (?, ?, ?)
        ON CONFLICT (device_uid) DO NOTHING
    """, (room_id, "ESP32-001", 2000))
    node_id = conn.execute(
        "SELECT id FROM node WHERE device_uid = ?", ("ESP32-001",)
    ).fetchone()[0]

    sensors = conn.execute(
        "SELECT id FROM sensor WHERE node_id = ? AND model = ?", (node_id, "DHT22")
    ).fetchall()
    if len(sensors) > 1:
        raise ValueError("У узла несколько DHT22. Нужно выбрать один датчик.")
    if sensors:
        sensor_id = sensors[0][0]
    else:
        sensor_id = conn.execute(
            "INSERT INTO sensor (node_id, model) VALUES (?, ?)", (node_id, "DHT22")
        ).lastrowid

    for metric, unit in [("temperature", "°C"), ("humidity", "%")]:
        conn.execute("""
            INSERT INTO channel (sensor_id, metric, unit) VALUES (?, ?, ?)
            ON CONFLICT (sensor_id, metric) DO NOTHING
        """, (sensor_id, metric, unit))

    valid_from = datetime.now(timezone.utc).isoformat(
        timespec="milliseconds"
    ).replace("+00:00", "Z")

    for metric, lower, upper in [("temperature", 20.0, 26.0), ("humidity", 40.0, 60.0)]:
        channel_id = conn.execute(
            "SELECT id FROM channel WHERE sensor_id = ? AND metric = ?", (sensor_id, metric)
        ).fetchone()[0]
        exists = conn.execute(
            "SELECT id FROM threshold WHERE channel_id = ?", (channel_id,)
        ).fetchone()
        if exists is None:
            conn.execute("""
                INSERT INTO threshold (channel_id, lower_bound, upper_bound, valid_from)
                VALUES (?, ?, ?, ?)
            """, (channel_id, lower, upper, valid_from))

print("Начальные данные подготовлены.")

## 4. Запросы
Семь вопросов к базе: где стоит узел, какие у него каналы, последние показания, история за интервал, статистика, действующие пороги и предупреждения. Даты интервала можно изменить.

In [ ]:
# Интервал демонстрации в UTC; при необходимости измени даты.
start = "2026-10-19T00:00:00.000Z"
end = "2026-10-20T00:00:00.000Z"
now = datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")

print("\n1. Комнаты и узлы")
rows = conn.execute("""
    SELECT room.name, room.area_m2, node.device_uid, node.period_ms FROM room INNER JOIN node ON node.room_id = room.id
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n2. Датчики и каналы")
rows = conn.execute("""
    SELECT node.device_uid, sensor.model, channel.id, channel.metric, channel.unit FROM node INNER JOIN sensor ON sensor.node_id = node.id INNER JOIN channel ON channel.sensor_id = sensor.id ORDER BY channel.id
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n3. Последнее значение каждого канала")
rows = conn.execute("""
    SELECT channel.metric, m.ts, m.value, channel.unit FROM measurement AS m INNER JOIN channel ON channel.id = m.channel_id WHERE m.ts = (SELECT MAX(m2.ts) FROM measurement AS m2 WHERE m2.channel_id = m.channel_id) ORDER BY channel.id
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n4. Измерения за интервал")
rows = conn.execute("""
    SELECT channel_id, ts, value FROM measurement WHERE ts >= ? AND ts < ? ORDER BY ts, channel_id
""", (start, end)).fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n5. Статистика за интервал")
rows = conn.execute("""
    SELECT channel.metric, COUNT(*) AS count, MIN(m.value) AS minimum, MAX(m.value) AS maximum, AVG(m.value) AS average FROM measurement AS m INNER JOIN channel ON channel.id = m.channel_id WHERE m.ts >= ? AND m.ts < ? GROUP BY channel.id, channel.metric ORDER BY channel.id
""", (start, end)).fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n6. Действующие пороги")
rows = conn.execute("""
    SELECT channel.metric, threshold.lower_bound, threshold.upper_bound, threshold.valid_from, threshold.valid_to FROM threshold INNER JOIN channel ON channel.id = threshold.channel_id WHERE valid_from <= ? AND (valid_to IS NULL OR ? < valid_to) ORDER BY channel.id
""", (now, now)).fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

print("\n7. Предупреждения с измерениями и границами")
rows = conn.execute("""
    SELECT channel.metric, alert.ts, measurement.value, threshold.lower_bound, threshold.upper_bound, alert.direction FROM alert INNER JOIN channel ON channel.id = alert.channel_id INNER JOIN measurement ON measurement.channel_id = alert.channel_id AND measurement.ts = alert.ts INNER JOIN threshold ON threshold.id = alert.threshold_id ORDER BY alert.ts DESC
""").fetchall()
if not rows:
    print("Нет записей для выбранных условий.")
for row in rows:
    print(dict(row))

## 5. Закрытие соединения

In [ ]:
conn.close()
print("Соединение закрыто.")